# Gold Layer

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


## Create Clean Dimension Table

## Customer Dimension Table

CReating the dataframe for all the Customer dimension Tables

In [0]:
cus1= spark.table("vivo.silver.cust_info")

In [0]:
cus2= spark.table("vivo.silver.cust_add_details")

In [0]:
cus3= spark.table("vivo.silver.cust_country")

## Joining all the customer dimension tables

In [0]:
customer= cus1.join(cus2,cus1.customer_key==cus2.CID,"left").join(cus3,cus1.customer_key==cus3.CID,"left").select(cus1["*"], cus2["BDATE"],cus2["GEN"],cus3["CNTRY"])

In [0]:
customer.show()

+-----------+------------+---------+--------+--------------+------+------------------------+----------+------+-------------+
|customer_id|customer_key|firstname|lastname|marital_status|gender|customer_onboarding_date|     BDATE|   GEN|        CNTRY|
+-----------+------------+---------+--------+--------------+------+------------------------+----------+------+-------------+
|      11000|  AW00011000|      Jon|    Yang|       Married|  Male|              2025-10-06|1971-10-06|  Male|    Australia|
|      11001|  AW00011001|   Eugene|   Huang|        Single|  Male|              2025-10-06|1976-05-10|  Male|    Australia|
|      11002|  AW00011002|    Ruben|  Torres|       Married|  Male|              2025-10-06|1971-02-09|  Male|    Australia|
|      11003|  AW00011003|  Christy|     Zhu|        Single|Female|              2025-10-06|1973-08-14|Female|    Australia|
|      11004|  AW00011004|Elizabeth| Johnson|        Single|Female|              2025-10-06|1979-08-05|Female|    Australia|


## Clean the Gender data

In [0]:
customer= customer.withColumn("New_Gender", when(col("gender") != "N/A", col("gender")).otherwise(coalesce(col("GEN"),col("gender") )))


In [0]:
customer= customer.drop("gender","GEN")

In [0]:
customer= customer.withColumnRenamed("New_Gender","gender")


## Creating a new Primary key column !

In [0]:
wind= Window.orderBy("customer_id")
customer= customer.withColumn("CustomerId", row_number().over(wind))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
customer.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------+---------+--------+--------------+------------------------+----------+-------------+------+----------+
|customer_id|customer_key|firstname|lastname|marital_status|customer_onboarding_date|     BDATE|        CNTRY|gender|CustomerId|
+-----------+------------+---------+--------+--------------+------------------------+----------+-------------+------+----------+
|      11000|  AW00011000|      Jon|    Yang|       Married|              2025-10-06|1971-10-06|    Australia|  Male|         1|
|      11001|  AW00011001|   Eugene|   Huang|        Single|              2025-10-06|1976-05-10|    Australia|  Male|         2|
|      11002|  AW00011002|    Ruben|  Torres|       Married|              2025-10-06|1971-02-09|    Australia|  Male|         3|
|      11003|  AW00011003|  Christy|     Zhu|        Single|              2025-10-06|1973-08-14|    Australia|Female|         4|
|      11004|  AW00011004|Elizabeth| Johnson|        Single|              2025-10-06|1979-08-05| 

## Ordering the columns

In [0]:
customer = customer.select("CustomerId","customer_id","customer_key","firstname","lastname",col("CNTRY").alias("country"),"BDATE","gender","marital_status","customer_onboarding_date")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
customer.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----------+------------+---------+--------+-------------+----------+------+--------------+------------------------+
|CustomerId|customer_id|customer_key|firstname|lastname|      country|     BDATE|gender|marital_status|customer_onboarding_date|
+----------+-----------+------------+---------+--------+-------------+----------+------+--------------+------------------------+
|         1|      11000|  AW00011000|      Jon|    Yang|    Australia|1971-10-06|  Male|       Married|              2025-10-06|
|         2|      11001|  AW00011001|   Eugene|   Huang|    Australia|1976-05-10|  Male|        Single|              2025-10-06|
|         3|      11002|  AW00011002|    Ruben|  Torres|    Australia|1971-02-09|  Male|       Married|              2025-10-06|
|         4|      11003|  AW00011003|  Christy|     Zhu|    Australia|1973-08-14|Female|        Single|              2025-10-06|
|         5|      11004|  AW00011004|Elizabeth| Johnson|    Australia|1979-08-05|Female|        S

## Creating the Customer Dimension Table

In [0]:
customer.write.mode("overwrite").option("overwriteSchema","true").option("header","true").saveAsTable("vivo.gold.dim_customer")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Create Product Dimension Table

Create the dataframes for all the product tables

In [0]:
pf1= spark.table("vivo.silver.prd_info")

In [0]:
pf2= spark.table("vivo.silver.product_category")

In [0]:
pf1.show()

+------+----------+-------+--------------------+--------+--------+------------+------------+
|prd_id|  prod_key|prod_id|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_date|
+------+----------+-------+--------------------+--------+--------+------------+------------+
|   601|   BB-7421|  CO_BB|   LL Bottom Bracket|      24|     N/A|  2013-07-01|        NULL|
|   602|   BB-8107|  CO_BB|   ML Bottom Bracket|      45|     N/A|  2013-07-01|        NULL|
|   603|   BB-9108|  CO_BB|   HL Bottom Bracket|      54|     N/A|  2013-07-01|        NULL|
|   478|   BC-M005|  AC_BC|Mountain Bottle Cage|       4|Mountain|  2013-07-01|        NULL|
|   479|   BC-R205|  AC_BC|    Road Bottle Cage|       3|    Road|  2013-07-01|        NULL|
|   596|BK-M18B-40|  BI_MB|Mountain-500 Blac...|     295|Mountain|  2013-07-01|        NULL|
|   597|BK-M18B-42|  BI_MB|Mountain-500 Blac...|     295|Mountain|  2013-07-01|        NULL|
|   598|BK-M18B-44|  BI_MB|Mountain-500 Blac...|     295|Mountain|  20

## Data cleaning

In [0]:
pf1= pf1.where(col("prd_end_date").isNull())

In [0]:
pf1.select("prod_key").groupBy("prod_key").count().where(col("count")>1).show()

+--------+-----+
|prod_key|count|
+--------+-----+
+--------+-----+



In [0]:
pf2.show()

+-----+-----------+-----------------+-----------+
|   ID|        CAT|           SUBCAT|MAINTENANCE|
+-----+-----------+-----------------+-----------+
|AC_BR|Accessories|       Bike Racks|        Yes|
|AC_BS|Accessories|      Bike Stands|         No|
|AC_BC|Accessories|Bottles and Cages|         No|
|AC_CL|Accessories|         Cleaners|        Yes|
|AC_FE|Accessories|          Fenders|         No|
|AC_HE|Accessories|          Helmets|        Yes|
|AC_HP|Accessories|  Hydration Packs|         No|
|AC_LI|Accessories|           Lights|        Yes|
|AC_LO|Accessories|            Locks|        Yes|
|AC_PA|Accessories|         Panniers|         No|
|AC_PU|Accessories|            Pumps|        Yes|
|AC_TT|Accessories|  Tires and Tubes|        Yes|
|BI_MB|      Bikes|   Mountain Bikes|        Yes|
|BI_RB|      Bikes|       Road Bikes|        Yes|
|BI_TB|      Bikes|    Touring Bikes|        Yes|
|CL_BS|   Clothing|       Bib-Shorts|         No|
|CL_CA|   Clothing|             Caps|         No|


## Join all the product dimension tables

In [0]:
product= pf1.join(pf2,pf1.prod_id==pf2.ID,"left").select(pf1["*"],pf2["CAT"],pf2["SUBCAT"],pf2["MAINTENANCE"])

In [0]:
product.show()

+------+----------+-------+--------------------+--------+--------+------------+------------+-----------+-----------------+-----------+
|prd_id|  prod_key|prod_id|              prd_nm|prd_cost|prd_line|prd_start_dt|prd_end_date|        CAT|           SUBCAT|MAINTENANCE|
+------+----------+-------+--------------------+--------+--------+------------+------------+-----------+-----------------+-----------+
|   601|   BB-7421|  CO_BB|   LL Bottom Bracket|      24|     N/A|  2013-07-01|        NULL| Components|  Bottom Brackets|        Yes|
|   602|   BB-8107|  CO_BB|   ML Bottom Bracket|      45|     N/A|  2013-07-01|        NULL| Components|  Bottom Brackets|        Yes|
|   603|   BB-9108|  CO_BB|   HL Bottom Bracket|      54|     N/A|  2013-07-01|        NULL| Components|  Bottom Brackets|        Yes|
|   478|   BC-M005|  AC_BC|Mountain Bottle Cage|       4|Mountain|  2013-07-01|        NULL|Accessories|Bottles and Cages|         No|
|   479|   BC-R205|  AC_BC|    Road Bottle Cage|       

## Create a new Primary key for the Product Dimension table

In [0]:
wind= Window.orderBy(col("prod_key"), col("prd_start_dt"))
product= product.withColumn("ProductID", row_number().over(wind))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Sorting of Columns

In [0]:
product= product.select("ProductID", "prd_id", "prod_id", "prod_key", "prd_nm", "CAT", "SUBCAT", "prd_cost", "MAINTENANCE", "prd_start_dt", "prd_line")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
product.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+------+-------+----------+--------------------+-----------+-----------------+--------+-----------+------------+--------+
|ProductID|prd_id|prod_id|  prod_key|              prd_nm|        CAT|           SUBCAT|prd_cost|MAINTENANCE|prd_start_dt|prd_line|
+---------+------+-------+----------+--------------------+-----------+-----------------+--------+-----------+------------+--------+
|        1|   601|  CO_BB|   BB-7421|   LL Bottom Bracket| Components|  Bottom Brackets|      24|        Yes|  2013-07-01|     N/A|
|        2|   602|  CO_BB|   BB-8107|   ML Bottom Bracket| Components|  Bottom Brackets|      45|        Yes|  2013-07-01|     N/A|
|        3|   603|  CO_BB|   BB-9108|   HL Bottom Bracket| Components|  Bottom Brackets|      54|        Yes|  2013-07-01|     N/A|
|        4|   478|  AC_BC|   BC-M005|Mountain Bottle Cage|Accessories|Bottles and Cages|       4|         No|  2013-07-01|Mountain|
|        5|   479|  AC_BC|   BC-R205|    Road Bottle Cage|Accessories|Bottle

## Creating the Product Dimension Table

In [0]:
product.write.mode("overwrite").option("header","true").option("inferSchema","true").saveAsTable("vivo.gold.dim_product")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


# Create the Fact Table

## Create the dataframe for the gold.Customer, gold.Product and silver.Sales tables

In [0]:
sf= spark.table("vivo.silver.sales_details")

In [0]:
pf= spark.table("vivo.gold.dim_product")

In [0]:
cf= spark.table("vivo.gold.dim_customer")

## Join all the dataframes

In [0]:
sales= sf.join(pf, sf.sls_prd_key==pf.prod_key, "left").join(cf, sf.sls_cust_id==cf.customer_id, "left").select(sf["*"], pf["ProductID"], cf["CustomerID"])

### ## Drop Unnecessary columns

In [0]:
sales= sales.drop("sls_prd_key","sls_cust_id")

## Sort the column Names

In [0]:
sales= sales.select("sls_ord_num", "CustomerID", "ProductID", "sls_order_dt", "sls_ship_dt", "sls_due_dt", "sls_sales", "sls_quantity", "sls_price")

In [0]:
sales.show()

+-----------+----------+---------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|CustomerID|ProductID|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+----------+---------+------------+-----------+----------+---------+------------+---------+
|    SO43697|     10769|       80|  2010-12-29| 2011-01-05|2011-01-10|     3578|           1|   3578.0|
|    SO43698|     17390|       36|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|   3400.0|
|    SO43699|     14864|       36|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|   3400.0|
|    SO43700|      3502|       47|  2010-12-29| 2011-01-05|2011-01-10|      699|           1|    699.0|
|    SO43701|         4|       36|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|   3400.0|
|    SO43702|     16646|       76|  2010-12-30| 2011-01-06|2011-01-11|     3578|           1|   3578.0|
|    SO43703|      5625|       80|  2010-12-30| 2011-01-06|2011-

## Create the Fact Sales Table

In [0]:
sales.write.mode("overwrite").option("header","true").option("inferSchema","true").saveAsTable("vivo.gold.fact_sales")

GOLD LAYER COMPLETED !!